In [1]:
from analytics.db import query

In [2]:
query("""---What percent of all reviews give 5 stars? Round to 1 decimal.
SELECT
    ROUND(100.0 * COUNT(*) FILTER (WHERE review_score = 5) / COUNT(*), 1) AS pct_5_star_reviews
FROM order_reviews;""")

,pct_5_star_reviews
0,57.8


In [3]:
query("""
      --way1
    SELECT 
        ROUND(SUM(i.price),2) AS way_1_items_revenue
    FROM order_items AS i
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
""")

,way_1_items_revenue
0,854686.33


In [4]:
query("""
      --way2
    SELECT 
        ROUND(SUM(p.payment_value),2) AS way_2_payments_revenue
    FROM order_payments AS p
    JOIN orders AS o ON o.order_id = p.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
""")

,way_2_payments_revenue
0,1022425.32


In [5]:
query("""
      --way3
    SELECT 
        p.payment_type,
        SUM(i.price) AS revenue
    FROM order_items AS i
    JOIN order_payments AS p ON p.order_id = i.order_id
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY p.payment_type
    ORDER BY revenue DESC
""")

,payment_type,revenue
0,credit_card,695780.67
1,boleto,118214.12
2,debit_card,39780.61
3,voucher,30882.00


In [6]:
query("""
    SELECT 
        SUM(i.price) AS way_3_total_revenue
    FROM order_items AS i
    JOIN order_payments AS p ON p.order_id = i.order_id
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
""")

,way_3_total_revenue
0,884657.4


In [7]:
august_reconciliation_query = """
    WITH item_agg AS (
        -- Group item prices and freight per order
        SELECT order_id, SUM(price) AS total_price, SUM(freight_value) AS total_freight
        FROM order_items
        GROUP BY order_id
    ),
    payment_agg AS (
        -- Group payments and count payment rows per order
        SELECT order_id, SUM(payment_value) AS total_payment, COUNT(*) AS payment_rows
        FROM order_payments
        GROUP BY order_id
    )
    -- Combine with August 2018 orders using LEFT JOIN
    SELECT 
        o.order_id,
        o.order_status,
        i.total_price,
        i.total_freight,
        p.total_payment,
        p.payment_rows
    FROM orders AS o
    LEFT JOIN item_agg AS i ON i.order_id = o.order_id
    LEFT JOIN payment_agg AS p ON p.order_id = o.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
"""

In [8]:
query(f"""
    WITH base AS ({august_reconciliation_query})
    SELECT
        SUM(total_price) AS way_1_items,
        SUM(total_freight) AS total_freight,
        SUM(CASE WHEN total_price IS NULL THEN total_payment ELSE 0 END) AS payments_with_no_items,
        SUM(COALESCE(total_payment, 0) - COALESCE(total_price, 0) - COALESCE(total_freight, 0)) AS small_differences,
        SUM(total_payment) AS way_2_payments
    FROM base
""")

,way_1_items,total_freight,payments_with_no_items,small_differences,way_2_payments
0,854686.33,148622.14,19194.21,19116.85,1022425.32


In [9]:
query("""
    WITH multi_pay AS (
        -- Find orders with more than one payment row
        SELECT order_id
        FROM order_payments
        GROUP BY order_id
        HAVING COUNT(*) > 1
    )
    SELECT 
        SUM(i.price) AS duplicated_item_prices
    FROM order_items AS i
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
      AND i.order_id IN (SELECT order_id FROM multi_pay)
""")

,duplicated_item_prices
0,23229.25


In [10]:
# 1. Confirmed Primary Figure (Products sold, excluding cancelled/unavailable items)
query("""
    SELECT 
        SUM(i.price) AS official_revenue
    FROM order_items AS i
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
      AND o.order_status NOT IN ('canceled', 'unavailable')
""")

,official_revenue
0,848860.1


In [11]:
# 2. Sensitivity of Options: Calculating the figure if shipping costs are included
query("""
    SELECT 
        SUM(i.price + i.freight_value) AS revenue_with_freight
    FROM order_items AS i
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
      AND o.order_status NOT IN ('canceled', 'unavailable')
""")

,revenue_with_freight
0,996973.51


In [12]:
# 3. Sensitivity of the options: Calculating the figure if we consider only delivered orders.
query("""
    SELECT 
        SUM(i.price) AS revenue_delivered_only
    FROM order_items AS i
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.order_purchase_timestamp >= '2018-08-01'
      AND o.order_purchase_timestamp < '2018-09-01'
      AND o.order_status = 'delivered'
""")

,revenue_delivered_only
0,838576.64
